# Residuals and Mass Step Correction

This notebook computes Hubble residuals and applies a mass-step correction.
Steps:
1. Load supernova dataset.
2. Compute residuals = MU - MU_MODEL.
3. Estimate mass step (gamma).
4. Apply mass-step correction.
5. Inspect results.

In [ ]:
import numpy as np
import pandas as pd

## 1. Load data
Replace the CSV path with your dataset. Must include columns:
- `MU`: observed distance modulus
- `MU_MODEL`: model distance modulus (from cosmology)
- `LOGMSTAR`: host galaxy log stellar mass
- (optional) `MUERR`: uncertainty on MU

In [ ]:
csv_path = "your_data.csv"
df = pd.read_csv(csv_path)
df.head()

## 2. Compute residuals

In [ ]:
df["RESIDUAL"] = pd.to_numeric(df["MU"], errors="coerce") - pd.to_numeric(df["MU_MODEL"], errors="coerce")
df[["MU", "MU_MODEL", "RESIDUAL"]].head()

## 3. Estimate mass step
Gamma is the weighted difference in mean residuals between high-mass (logM* ≥ 10) and low-mass hosts.

In [ ]:
def weighted_mean(x, w):
    good = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not np.any(good):
        return np.nan, np.nan
    W = np.sum(w[good])
    m = np.sum(w[good] * x[good]) / W
    se = np.sqrt(1.0 / W)
    return m, se

resid = pd.to_numeric(df["RESIDUAL"], errors="coerce").to_numpy()
mass  = pd.to_numeric(df["LOGMSTAR"], errors="coerce").to_numpy()
sigma = pd.to_numeric(df["MUERR"], errors="coerce").to_numpy() if "MUERR" in df.columns else np.ones_like(resid)
w = 1.0 / sigma**2

hi = mass >= 10.0
lo = mass < 10.0

mean_hi, se_hi = weighted_mean(resid[hi], w[hi])
mean_lo, se_lo = weighted_mean(resid[lo], w[lo])
gamma = mean_hi - mean_lo
stderr = np.sqrt(se_hi**2 + se_lo**2)

print(f"Mass step γ = {gamma:.4f} ± {stderr:.4f}")

## 4. Apply mass-step correction
Subtract gamma for high-mass hosts so the corrected residuals are aligned.

In [ ]:
df["RESIDUAL_CORR"] = df["RESIDUAL"] - gamma * (df["LOGMSTAR"] >= 10.0).astype(float)
df[["RESIDUAL", "RESIDUAL_CORR", "LOGMSTAR"]].head()

## 5. Inspect results
You can now analyze distributions or plot residuals before/after correction.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.hist(df["RESIDUAL"], bins=30, alpha=0.5, label="Residuals")
plt.hist(df["RESIDUAL_CORR"], bins=30, alpha=0.5, label="Corrected")
plt.legend()
plt.xlabel("Residual (mag)")
plt.ylabel("Count")
plt.show()